In [1]:
import numpy as np
import finesse
import matplotlib.pyplot as plt
from utils.finesse_base import base_kat
from finesse.knm import Map
from finesse.utilities.maps import circular_aperture

## Finesse Testing Ground

In [2]:
kat = base_kat.deepcopy()

ITM_ROC = -1934
ETM_ROC = 2245
initial_guess = -1
ETM_ROC = ETM_ROC + ETM_ROC*.2
ITM_ROC = ITM_ROC + ITM_ROC*0
print(f"ITM_ROC: {ITM_ROC}")
print(f"ETM_ROC: {ETM_ROC}")

kat.ITM.Rc = ITM_ROC
kat.ETM.Rc = ETM_ROC
x = y = np.linspace(-0.17, 0.17, 100)

kat.ETM.surface_map = Map(x, y, amplitude=circular_aperture(x,y,0.17))
# kat.ETM.phi = initial_guess

out = kat.run("run_locks(display_progress=true,pre_step=print_model_attr(ETM.phi))")


print(f"\nThe ETM tunning AFTER the successful lock is {kat.ETM.phi.value} deg")
print("g-value is ", kat.cavArm.g[0])


ITM_ROC: -1934
ETM_ROC: 2694.0
Error Signal Residuals at Each Iteration (W):
                         lock_length  
Iteration Number   0   ETM.phi=0.0
    1.16e-11   
Iteration Number   1   ETM.phi=-1.2387020549224899e-11
    7.67e-12   
Iteration Number   2   ETM.phi=-2.057792448732133e-11
    5.07e-12   
Iteration Number   3   ETM.phi=-2.599416037022081e-11
    3.36e-12   
Iteration Number   4   ETM.phi=-2.9575633077324025e-11
    2.22e-12   
Iteration Number   5   ETM.phi=-3.194387731860141e-11
    1.47e-12   
Iteration Number   6   ETM.phi=-3.3509889466551614e-11
    9.70e-13   
The ETM tunning AFTER the successful lock is -3.3509889466551614e-11 deg
g-value is  0.5142953203487928


In [6]:
sol = kat.run( """
    series(
        eigenmodes(cavArm, 0,       name="c0"), 
    )
    """)
gamma_list = abs(sol["c0"].eigvalues)
total_loss = 0
for gamma in gamma_list:
    total_loss += 1 - gamma**2 - 0.014
print(total_loss)


0.000531088714914767


In [9]:
kat_ideal = base_kat.deepcopy()
ref_power = kat_ideal.run()["circ"]
print(ref_power)
kat_aperture = base_kat.deepcopy()
kat_aperture.ETM.surface_map = Map(x, y, amplitude=circular_aperture(x,y,0.17))
aperture_power = kat_aperture.run()["circ"]
print(aperture_power)
print((1 - (aperture_power/ref_power))*100)



282.0955214139097
282.0710184681937
0.008686045632055883


In [5]:
86.86045632011474

86.86045632011474

In [16]:
from perturbation import calc_perturbed_params
ETM_ROC_perturbed, ITM_ROC_perturbed, Cav_L_perturbed = calc_perturbed_params(999, 999, 4999)
print(f"ETM_ROC: {ETM_ROC_perturbed}")
print(f"ITM_ROC: {ITM_ROC_perturbed}")
print(f"Cav_L: {Cav_L_perturbed}")
results = []
for ETM_ROC_p, ITM_ROC_p, Cav_L_p in zip(ETM_ROC_perturbed, ITM_ROC_perturbed, Cav_L_perturbed):
        result = [ETM_ROC_p, ITM_ROC_p, Cav_L_p]
        results.append(result)
print(results)

ETM_ROC: (np.float64(999.8026991692033), 999, np.float64(998.1973008307967))
ITM_ROC: (np.float64(1007.3422554907035), 999, np.float64(990.6577445092965))
Cav_L: (np.float64(4999.002681741694), 4999, np.float64(4998.997318258306))
[[np.float64(999.8026991692033), np.float64(1007.3422554907035), np.float64(4999.002681741694)], [999, 999, 4999], [np.float64(998.1973008307967), np.float64(990.6577445092965), np.float64(4998.997318258306)]]
